<a href="https://colab.research.google.com/github/youkjang/gefs_temperature_bias_correction/blob/main/notebooks/03_0_prepare_gefs_gfs_t2m_for_ml_bias_correction.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Prepare a Matched GEFS and GFS 2-m Temperature Dataset for ML Bias Correction

## 1. Introduction
- This notebook prepares the matched forecast–analysis dataset used in the GEFS 2-m temperature bias-correction workflow.

- The purpose is to load GEFS ensemble-mean 2-m temperature forecasts and matching GFS analysis fields, align them on the same spatial grid, and save the result as a NetCDF file for later bias-correction experiments.

- The saved dataset will be used by later notebooks for Machine-learning-based bias correction

## 2. Install and import packages

In [1]:
!pip -q install "eccodes>=2.37.0" cfgrib s3fs "fsspec==2025.3.0"

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.1/56.1 kB 2.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 91.6/91.6 kB 6.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 49.1/49.1 kB 3.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 87.3/87.3 kB 6.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.0/9.0 MB 74.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 7.2/7.2 MB 99.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 14.3/14.3 MB 86.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 88.0/88.0 kB 6.1 MB/s eta 0:00:00


In [2]:
!pip install git+https://github.com/youkjang/gefs_temperature_bias_correction.git

  Cloning https://github.com/youkjang/gefs_temperature_bias_correction.git to /tmp/pip-req-build-qfv324oj
  Running command git clone --filter=blob:none --quiet https://github.com/youkjang/gefs_temperature_bias_correction.git /tmp/pip-req-build-qfv324oj
  Resolved https://github.com/youkjang/gefs_temperature_bias_correction.git to commit 517eb14b43bd2b787a8a353eb0496975ecf776f0
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 79.5/79.5 kB 3.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 118.2/118.2 kB 8.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.1/10.1 MB 85.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.9/10.9 MB 95.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.6/1.6 MB 61.4 MB/s eta 0:00:00
  Created wheel for gefs-temperature-bias-correction: filename=gefs_temperature_bias_corr

In [3]:
from pathlib import Path

import numpy as np
import pandas as pd
import xarray as xr
import matplotlib.pyplot as plt

from gefs_bias_correction.config import ProjectConfig, Region
from gefs_bias_correction.synthetic import make_synthetic_t2m_dataset
from gefs_bias_correction.data_access import build_matched_t2m_dataset
from gefs_bias_correction.split import split_train_test_by_init_date
from gefs_bias_correction.mean_bias import (
    compute_mean_bias_by_lead,
    apply_mean_bias_correction,
)
from gefs_bias_correction.quantile_mapping import (
    fit_quantile_mapping_by_lead,
    apply_quantile_mapping_correction,
    make_quantile_mapping_summary_table,
)
from gefs_bias_correction.verification import summarize_by_lead

## 3. User configuration

- 92 GEFS initialization dates from **June 1, 2023 to August 31, 2023**
- five forecast lead times: f024, f048, f072, f096, and f120.



In [5]:
USE_SYNTHETIC = False  # Set True only for quick debugging.
N_QUANTILES = 101
TRAIN_FRACTION = 0.7

OUTPUT_DIR = Path("outputs")
FIGURE_DIR = OUTPUT_DIR / "figures"
TABLE_DIR = OUTPUT_DIR / "tables"
NETCDF_DIR = OUTPUT_DIR / "netcdf"

for directory in [FIGURE_DIR, TABLE_DIR, NETCDF_DIR]:
    directory.mkdir(parents=True, exist_ok=True)

config = ProjectConfig(
    init_dates=pd.date_range(
        "2023-06-01",
        "2023-08-31",
        freq="D",
    ).strftime("%Y%m%d").tolist(),
    init_hour="00",
    forecast_hours=[24, 48, 72, 96, 120],
    train_fraction=TRAIN_FRACTION,

    region=Region(
        name="CONUS",
        west=-130,
        east=-60,
        south=20,
        north=55,
    ),

    # region = Region(
    # name="Southwest_US",
    # west=-125,
    # east=-100,
    # south=25,
    # north=42,
    # ),

    # region = Region(
    # name="Southern_Plains",
    # west=-107,
    # east=-90,
    # south=25,
    # north=40,
    # ),

    # region = Region(
    # name="Northeast_US",
    # west=-85,
    # east=-66,
    # south=36,
    # north=48,
    # ),

    cache_dir="data/herbie_cache",
    verbose=True,
    skip_missing=True,
)

config

ProjectConfig(init_dates=['20230601', '20230602', '20230603', '20230604', '20230605', '20230606', '20230607', '20230608', '20230609', '20230610', '20230611', '20230612', '20230613', '20230614', '20230615', '20230616', '20230617', '20230618', '20230619', '20230620', '20230621', '20230622', '20230623', '20230624', '20230625', '20230626', '20230627', '20230628', '20230629', '20230630', '20230701', '20230702', '20230703', '20230704', '20230705', '20230706', '20230707', '20230708', '20230709', '20230710', '20230711', '20230712', '20230713', '20230714', '20230715', '20230716', '20230717', '20230718', '20230719', '20230720', '20230721', '20230722', '20230723', '20230724', '20230725', '20230726', '20230727', '20230728', '20230729', '20230730', '20230731', '20230801', '20230802', '20230803', '20230804', '20230805', '20230806', '20230807', '20230808', '20230809', '20230810', '20230811', '20230812', '20230813', '20230814', '20230815', '20230816', '20230817', '20230818', '20230819', '20230820', '2

## 4. Load the matched forecast-analysis dataset

VERBOSE = False:  to remove many print lines to upload a cleaner notebook to GitHub after running this notebook

In [18]:
VERBOSE = False

if USE_SYNTHETIC:
    ds = make_synthetic_t2m_dataset(config)
else:
  if VERBOSE:
    ds = build_matched_t2m_dataset(config)

ds

<xarray.Dataset> Size: 55MB
Dimensions:              (case: 460, latitude: 71, longitude: 141)
Coordinates:
  * case                 (case) int64 4kB 0 1 2 3 4 5 ... 455 456 457 458 459
  * latitude             (latitude) float64 568B 20.0 20.5 21.0 ... 54.5 55.0
  * longitude            (longitude) float64 1kB -130.0 -129.5 ... -60.5 -60.0
    gribfile_projection  object 8B None
    init_date            (case) datetime64[us] 4kB 2023-06-01 ... 2023-08-31
    fhr                  (case) int64 4kB 24 48 72 96 120 24 ... 24 48 72 96 120
    valid_time           (case) datetime64[us] 4kB 2023-06-02 ... 2023-09-05
Data variables:
    forecast_t2m_c       (case, latitude, longitude) float32 18MB 20.65 ... 1...
    analysis_t2m_c       (case, latitude, longitude) float64 37MB 20.61 ... 1...
Attributes:
    description:        Matched GEFS ensemble-mean 2-m temperature forecasts ...
    case_definition:    Each case is one initialization date and one forecast...
    temperature_units:  degrees Celsius
    forecast_variable:  forecast_t2m_c
    analysis_variable:  analysis_t2m_c

## 5. Save matched_gefs_gfs_t2m_for_ml_bias_correction.nc

In [7]:
# Add useful metadata
ds.attrs["description"] = (
    "Matched GEFS ensemble-mean 2-m temperature forecasts and GFS analysis fields "
    "for machine-learning-based temperature bias correction."
)
ds.attrs["case_definition"] = "Each case is one initialization date and one forecast hour."
ds.attrs["temperature_units"] = "degrees Celsius"
ds.attrs["forecast_variable"] = "forecast_t2m_c"
ds.attrs["analysis_variable"] = "analysis_t2m_c"


In [8]:
# Output file name
output_path = NETCDF_DIR/ "matched_gefs_gfs_t2m_for_ml_bias_correction.nc"

# Save as NetCDF
ds.to_netcdf(output_path)

In [9]:
ds

<xarray.Dataset> Size: 55MB
Dimensions:              (case: 460, latitude: 71, longitude: 141)
Coordinates:
  * case                 (case) int64 4kB 0 1 2 3 4 5 ... 455 456 457 458 459
  * latitude             (latitude) float64 568B 20.0 20.5 21.0 ... 54.5 55.0
  * longitude            (longitude) float64 1kB -130.0 -129.5 ... -60.5 -60.0
    gribfile_projection  object 8B None
    init_date            (case) datetime64[us] 4kB 2023-06-01 ... 2023-08-31
    fhr                  (case) int64 4kB 24 48 72 96 120 24 ... 24 48 72 96 120
    valid_time           (case) datetime64[us] 4kB 2023-06-02 ... 2023-09-05
Data variables:
    forecast_t2m_c       (case, latitude, longitude) float32 18MB 20.65 ... 1...
    analysis_t2m_c       (case, latitude, longitude) float64 37MB 20.61 ... 1...
Attributes:
    description:        Matched GEFS ensemble-mean 2-m temperature forecasts ...
    case_definition:    Each case is one initialization date and one forecast...
    temperature_units:  degrees Celsius
    forecast_variable:  forecast_t2m_c
    analysis_variable:  analysis_t2m_c

## 6. Check: read the save file and compare with ds

In [14]:
ds_read=xr.open_dataset(output_path)

In [15]:
ds_read=ds_read.load()

In [16]:
ds_read

<xarray.Dataset> Size: 55MB
Dimensions:              (case: 460, latitude: 71, longitude: 141)
Coordinates:
  * case                 (case) int64 4kB 0 1 2 3 4 5 ... 455 456 457 458 459
  * latitude             (latitude) float64 568B 20.0 20.5 21.0 ... 54.5 55.0
  * longitude            (longitude) float64 1kB -130.0 -129.5 ... -60.5 -60.0
    gribfile_projection  float64 8B nan
    init_date            (case) datetime64[ns] 4kB 2023-06-01 ... 2023-08-31
    fhr                  (case) int64 4kB 24 48 72 96 120 24 ... 24 48 72 96 120
    valid_time           (case) datetime64[ns] 4kB 2023-06-02 ... 2023-09-05
Data variables:
    forecast_t2m_c       (case, latitude, longitude) float32 18MB 20.65 ... 1...
    analysis_t2m_c       (case, latitude, longitude) float64 37MB 20.61 ... 1...
Attributes:
    description:        Matched GEFS ensemble-mean 2-m temperature forecasts ...
    case_definition:    Each case is one initialization date and one forecast...
    temperature_units:  degrees Celsius
    forecast_variable:  forecast_t2m_c
    analysis_variable:  analysis_t2m_c

In [17]:
if ds.identical(ds_read):
    print("The datasets ds and ds_read are identical.")
else:
    print("The datasets ds and ds_read are NOT identical.\n")
    # Identify differences in data variables if they exist
    difference = ds - ds_read
    print("Differences in data variables (if any):")
    print(difference)

The datasets ds and ds_read are identical.


###Note on AI Assistance

This notebook was developed with AI. See the repository README for details.